# lltTinyStories sweep (Run Library)

Two training queues on Kaggle's 2x T4, one per GPU, over one shared token cache.

In [ ]:

import json, os, platform, subprocess, sys, time
T_SESSION = time.time()
print("python", platform.python_version())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "tokenizers"], check=False)


In [ ]:
PREP_SRC = '"""Data prep for the lltTinyStories sweep: train the BPE tokenizer on the TRAIN split only, encode\nboth splits once, and cache the token ids as uint16 .npy files that every training run reuses.\n\nUsage: python prep_tinystories.py <cache_dir> <work_dir> [max_train_stories]\n"""\nimport json, os, sys, time\nimport numpy as np\n\ncache_dir, work_dir = sys.argv[1], sys.argv[2]\nmax_train = int(sys.argv[3]) if len(sys.argv) > 3 else None\nos.makedirs(cache_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)\nfrom datasets import load_dataset\nfrom tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders\n\nT0 = time.time()\nds = load_dataset("roneneldan/TinyStories")\ntrain_text, val_text = ds["train"]["text"], ds["validation"]["text"]\nif max_train: train_text = train_text[:max_train]\nload_sec = time.time() - T0\n\ntok = Tokenizer(models.BPE())\ntok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\ntok.decoder = decoders.ByteLevel()\ntrainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<|eot|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\nt = time.time(); tok.train_from_iterator(train_text[:300000], trainer=trainer); tok_sec = time.time() - t\nEOT = tok.token_to_id("<|eot|>")\ntok.save(os.path.join(cache_dir, "tokenizer.json")); tok.save(os.path.join(work_dir, "tokenizer.json"))\n\nenc = getattr(tok, "encode_batch_fast", None) or tok.encode_batch\n\n\ndef encode_all(texts):\n    out = []\n    for e in enc(texts):\n        out.append(np.asarray(e.ids, dtype=np.uint16)); out.append(np.array([EOT], dtype=np.uint16))\n    return np.concatenate(out)\n\n\nt = time.time()\ntrain_ids = np.concatenate([encode_all(train_text[i:i + 100000]) for i in range(0, len(train_text), 100000)])\nval_ids = encode_all(val_text)\nencode_sec = time.time() - t\n\nhalf = len(val_text) // 2\nval_b_start = int(encode_all(val_text[:half]).size)\n\n# Exact byte length of every token id. Byte-level BPE maps each byte to one printable character\n# (the GPT-2 table), so a token\'s characters map back to its bytes one for one.\nbs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))\ncs = bs[:]; n = 0\nfor b in range(256):\n    if b not in bs: bs.append(b); cs.append(256 + n); n += 1\nchar_to_byte = {chr(c): b for b, c in zip(bs, cs)}\nV = tok.get_vocab_size()\ntoken_bytes = np.zeros(V, dtype=np.int64)\nfor i in range(V):\n    s = tok.id_to_token(i)\n    token_bytes[i] = 1 if i == EOT else len(s)\n    assert i == EOT or all(ch in char_to_byte for ch in s), s\n# Check: token bytes over the val split equal the UTF-8 bytes of the text, plus one per story.\nval_bytes = sum(len(s.encode()) for s in val_text)\nassert int(token_bytes[val_ids].sum()) == val_bytes + len(val_text), (int(token_bytes[val_ids].sum()), val_bytes)\n\nt = time.time()\nnp.save(os.path.join(cache_dir, "train_ids.npy"), train_ids); np.save(os.path.join(cache_dir, "val_ids.npy"), val_ids)\nnp.save(os.path.join(cache_dir, "token_bytes.npy"), token_bytes)\nsave_sec = time.time() - t\nt = time.time(); _ = np.load(os.path.join(cache_dir, "train_ids.npy")); _ = np.load(os.path.join(cache_dir, "val_ids.npy")); load_cache_sec = time.time() - t\n\nmeta = {"eot": EOT, "valBStartToken": val_b_start,\n        "data": {"dataset": "roneneldan/TinyStories", "trainStories": len(train_text), "valStories": len(val_text),\n                 "trainTokens": int(train_ids.size), "valTokens": int(val_ids.size),\n                 "valAStories": half, "valBStartToken": val_b_start,\n                 "bytesPerToken": round(val_bytes / (val_ids.size - len(val_text)), 3)},\n        "timing": {"datasetLoadSec": round(load_sec, 1), "tokenizerTrainSec": round(tok_sec, 1),\n                   "encodeSec": round(encode_sec, 1), "encoder": enc.__name__,\n                   "cacheSaveSec": round(save_sec, 1), "cacheLoadSec": round(load_cache_sec, 1),\n                   "cpuCount": os.cpu_count()}}\njson.dump(meta, open(os.path.join(cache_dir, "meta.json"), "w"), indent=1)\njson.dump(meta, open(os.path.join(work_dir, "prep_meta.json"), "w"), indent=1)\nprint(json.dumps(meta, indent=1))\n'
open('/tmp/prep.py', 'w').write(PREP_SRC)
WORKER_SRC = '"""Training worker for the lltTinyStories sweep (kernel llt-tinystories-sweep).\n\nRuns a queue of configs on ONE GPU (the launcher sets CUDA_VISIBLE_DEVICES before this process\nimports torch). Reads token ids from a shared .npy cache, so the corpus is encoded once per\nsession, not once per run. Writes one Run Library JSON per run as soon as that run ends.\n\nUsage: python worker_tinystories.py <queue.json> <cache_dir> <out_dir>\n"""\nimport json, math, os, platform, sys, time\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nqueue_path, cache_dir, out_dir = sys.argv[1:4]\nQUEUE = json.load(open(queue_path))\nDEV = "cuda" if torch.cuda.is_available() else "cpu"\nif DEV == "cpu": torch.set_num_threads(4)\nCAP = torch.cuda.get_device_capability(0) if DEV == "cuda" else (0, 0)\n# Pick the dtype from the hardware, not from torch.cuda.is_bf16_supported(): that call returns True\n# on a T4, which has no bf16 tensor cores (measured, run llt-gpu-smoke).\nAMP_DTYPE = torch.bfloat16 if CAP >= (8, 0) else torch.float16\nUSE_SCALER = DEV == "cuda" and AMP_DTYPE == torch.float16\n\ntrain_ids = np.load(os.path.join(cache_dir, "train_ids.npy"), mmap_mode="r")\nval_ids = np.load(os.path.join(cache_dir, "val_ids.npy"))\ntok_bytes = np.load(os.path.join(cache_dir, "token_bytes.npy"))  # bytes per token id (EOT counts 1)\nmeta = json.load(open(os.path.join(cache_dir, "meta.json")))\nVAL_B_START = meta["valBStartToken"]\nEOT = meta["eot"]\n\nfrom tokenizers import Tokenizer\ntok = Tokenizer.from_file(os.path.join(cache_dir, "tokenizer.json"))\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps=1e-5):\n        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps\n    def forward(self, x):\n        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)\n\n\ndef rope_tables(T, hd, device):\n    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))\n    f = torch.outer(torch.arange(T, device=device).float(), inv)\n    return f.cos(), f.sin()\n\n\ndef apply_rope(x, cos, sin):\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)\n    return out.flatten(-2).type_as(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h):\n        super().__init__(); self.h = h\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.qkv = nn.Linear(d, 3 * d, bias=False); self.o = nn.Linear(d, d, bias=False)\n        hid = int(8 * d / 3 / 64 + 0.999) * 64\n        self.w1 = nn.Linear(d, hid, bias=False); self.w3 = nn.Linear(d, hid, bias=False); self.w2 = nn.Linear(hid, d, bias=False)\n    def forward(self, x, cos, sin):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))\n        h = self.n2(x)\n        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))\n\n\nclass GPT(nn.Module):\n    def __init__(self, V, d, L, H, ctx):\n        super().__init__()\n        self.emb = nn.Embedding(V, d); self.blocks = nn.ModuleList([Block(d, H) for _ in range(L)])\n        self.norm = RMSNorm(d); self.head = nn.Linear(d, V, bias=False); self.head.weight = self.emb.weight\n        self.H, self.ctx = H, ctx\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                std = 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02\n                nn.init.normal_(p, 0.0, std)\n    def forward(self, idx):\n        T = idx.shape[1]\n        cos, sin = rope_tables(T, self.emb.embedding_dim // self.H, idx.device)\n        x = self.emb(idx)\n        for b in self.blocks: x = b(x, cos, sin)\n        return self.head(self.norm(x))\n\n\ndef autocast():\n    return torch.autocast(DEV, dtype=AMP_DTYPE, enabled=DEV == "cuda")\n\n\ndef run_one(C):\n    seed = C["seed"]\n    torch.manual_seed(seed); np.random.seed(seed)\n    ctx, B, micro = C["ctx"], C["batch"], C.get("micro", C["batch"])\n    accum = B // micro\n    model = GPT(C["vocab"], C["d"], C["layers"], C["heads"], ctx).to(DEV)\n    n_params = sum(p.numel() for p in model.parameters())\n    n_emb = model.emb.weight.numel()\n    RUN = {"runId": C["runId"], "module": "lltTinyStories", "tier": C.get("tier", "T2"), "label": C["label"],\n           "hardware": {"gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else None,\n                        "count": torch.cuda.device_count() if DEV == "cuda" else 0,\n                        "vramGB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if DEV == "cuda" else None,\n                        "capability": list(CAP), "concurrent": C.get("concurrent", "one run per GPU, two GPUs busy")},\n           "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n                   "ampDtype": str(AMP_DTYPE).replace("torch.", ""), "gradScaler": USE_SCALER},\n           "config": {k: v for k, v in C.items() if k not in ("runId", "label", "tier")},\n           "seed": seed, "params": n_params, "nonEmbeddingParams": n_params - n_emb,\n           "flopsPerToken6N": 6 * n_params, "metrics": [], "trainCurve": [], "samples": [],\n           "data": meta["data"], "date": time.strftime("%Y-%m-%d")}\n\n    def batch():\n        ix = np.random.randint(0, train_ids.size - ctx - 1, micro)\n        x = torch.from_numpy(np.stack([np.asarray(train_ids[i:i + ctx], dtype=np.int64) for i in ix]))\n        y = torch.from_numpy(np.stack([np.asarray(train_ids[i + 1:i + 1 + ctx], dtype=np.int64) for i in ix]))\n        return x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)\n\n    @torch.no_grad()\n    def evaluate(start):\n        # Same window as the baseline: 640 sequences of ctx tokens from `start`. Bits per byte use\n        # the exact byte length of every target token.\n        model.eval(); nll, ntok, nbytes = 0.0, 0, 0\n        n = C["eval_batches"] * 16\n        for j in range(start, start + n * ctx, ctx * 16):\n            starts = [s for s in range(j, j + 16 * ctx, ctx) if s + ctx + 1 <= val_ids.size]\n            if not starts: break\n            x = torch.tensor(np.stack([val_ids[s:s + ctx].astype(np.int64) for s in starts]), device=DEV)\n            y_np = np.stack([val_ids[s + 1:s + ctx + 1].astype(np.int64) for s in starts])\n            y = torch.tensor(y_np, device=DEV)\n            with autocast():\n                logits = model(x)\n            nll += F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1), reduction="sum").item()\n            ntok += y.numel(); nbytes += int(tok_bytes[y_np].sum())\n        model.train()\n        return nll / ntok, nll / (math.log(2) * nbytes), ntok, nbytes\n\n    @torch.no_grad()\n    def sample(prompt, n=80, temp=0.8, topk=50):\n        model.eval(); g = torch.Generator(device=DEV).manual_seed(1234)\n        x = torch.tensor([tok.encode(prompt).ids], device=DEV)\n        for _ in range(n):\n            with autocast():\n                lg = model(x[:, -ctx:])[:, -1, :].float() / temp\n            v, _ = torch.topk(lg, topk); lg[lg < v[:, [-1]]] = -float("inf")\n            nxt = torch.multinomial(F.softmax(lg, -1), 1, generator=g)\n            if nxt.item() == EOT: break\n            x = torch.cat([x, nxt], 1)\n        model.train()\n        return tok.decode(x[0].tolist())\n\n    def lr_at(s, total, peak, warmup, offset=0, rewarm=0):\n        if s < warmup: lr = peak * (s + 1) / warmup\n        else:\n            p = (s - warmup) / max(1, total - warmup)\n            lr = peak * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))\n        if rewarm and s - offset < rewarm: lr = lr * (s - offset + 1) / rewarm\n        return lr\n\n    def make_opt():\n        decay = [p for n, p in model.named_parameters() if p.dim() >= 2]\n        no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]\n        return torch.optim.AdamW([{"params": decay, "weight_decay": C["wd"]}, {"params": no_decay, "weight_decay": 0.0}],\n                                 lr=C["lr"], betas=(0.9, 0.95), fused=DEV == "cuda")\n\n    def safe(v):\n        return None if (v is None or not math.isfinite(v)) else round(v, 4)\n\n    def train(start_step, end_step, opt, scaler, record, rewarm=0, save_at=None):\n        t_last = time.time(); tok_since = 0; run_loss = 0.0; run_n = 0; run_gn = 0.0; bad = 0\n        t0 = time.time()\n        for step in range(start_step, end_step + 1):\n            if record and (step % C["eval_every"] == 0 or step == end_step):\n                vl, bpb, ntok, nbytes = evaluate(0)\n                vlb, bpbb, _, _ = evaluate(VAL_B_START)\n                m = {"step": step, "tokens": step * B * ctx, "valLoss": safe(vl), "valBpb": safe(bpb),\n                     "valBLoss": safe(vlb), "valBBpb": safe(bpbb), "lr": lr_at(step, C["steps"], C["lr"], C["warmup"], start_step, rewarm),\n                     "elapsedSec": round(time.time() - t0, 1)}\n                if DEV == "cuda": m["memGB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n                if step > start_step: m["tokPerSec"] = int(tok_since / (time.time() - t_last))\n                record["metrics"].append(m); print(C["runId"], m, flush=True)\n                record.setdefault("evalTokens", ntok); record.setdefault("evalBytes", nbytes)\n                t_last = time.time(); tok_since = 0\n                if not math.isfinite(vl) or (step >= 1000 and vl > 6.0):\n                    bad += 1\n                    if bad >= 2 and C.get("stopIfDiverged"):\n                        record["stoppedEarly"] = step; break\n                if step == end_step or step % 1000 == 0:\n                    record["samples"] += [{"step": step, "prompt": p, "completion": sample(p)} for p in C["prompts"]]\n            if save_at is not None and step == save_at:\n                torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, os.path.join("/tmp", C["runId"] + f"-step{step}.pt"))\n            if step == end_step: break\n            for gparam in opt.param_groups: gparam["lr"] = lr_at(step, C["steps"], C["lr"], C["warmup"], start_step, rewarm)\n            opt.zero_grad(set_to_none=True)\n            tot = 0.0\n            for _ in range(accum):\n                x, y = batch()\n                with autocast():\n                    logits = model(x)\n                    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1)) / accum\n                (scaler.scale(loss) if scaler else loss).backward(); tot += loss.item()\n            if scaler: scaler.unscale_(opt)\n            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), C["clip"]).item()\n            if scaler: scaler.step(opt); scaler.update()\n            else: opt.step()\n            run_loss += tot; run_n += 1; run_gn += gn if math.isfinite(gn) else 0.0\n            if record and run_n == 25:\n                record["trainCurve"].append({"step": step + 1, "trainLoss": safe(run_loss / 25), "gradNorm": safe(run_gn / 25)})\n                run_loss = 0.0; run_n = 0; run_gn = 0.0\n            tok_since += B * ctx\n        return time.time() - t0\n\n    if DEV == "cuda": torch.cuda.reset_peak_memory_stats()\n    opt = make_opt(); scaler = torch.amp.GradScaler("cuda") if USE_SCALER else None\n    RUN["trainSec"] = round(train(0, C["steps"], opt, scaler, RUN, save_at=C.get("saveAt")), 1)\n    if DEV == "cuda": RUN["peakMemGB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n    if C.get("saveFinal"):\n        torch.save({"model": {k: v.half() for k, v in model.state_dict().items()}, "cfg": C},\n                   os.path.join(out_dir, C["runId"] + "-final-fp16.pt"))\n    with open(os.path.join(out_dir, C["runId"] + ".json"), "w") as f: json.dump(RUN, f, indent=1)\n    print("WROTE", C["runId"], RUN["metrics"][-1], flush=True)\n\n    # Resume tests for the "start from our checkpoint" exit: load model weights only (no optimizer\n    # state, which is what our published baseline checkpoints hold) and finish the last stretch.\n    for rw in C.get("resumeTests", []):\n        sd = torch.load(os.path.join("/tmp", C["runId"] + f"-step{C[\'saveAt\']}.pt"))\n        model.load_state_dict(sd)\n        torch.manual_seed(seed + 1000); np.random.seed(seed + 1000)\n        R = {"runId": C["runId"] + f"-resume{C[\'saveAt\']}-rewarm{rw}", "module": "lltTinyStories", "tier": "T2",\n             "label": f"ablation:resume=weights-only,rewarm={rw}", "hardware": RUN["hardware"], "env": RUN["env"],\n             "config": dict(RUN["config"], resumeFrom=C["saveAt"], rewarm=rw), "seed": seed, "params": n_params,\n             "metrics": [], "trainCurve": [], "samples": [], "date": time.strftime("%Y-%m-%d")}\n        opt = make_opt(); scaler = torch.amp.GradScaler("cuda") if USE_SCALER else None\n        R["trainSec"] = round(train(C["saveAt"], C["steps"], opt, scaler, R, rewarm=rw), 1)\n        with open(os.path.join(out_dir, R["runId"] + ".json"), "w") as f: json.dump(R, f, indent=1)\n        print("WROTE", R["runId"], R["metrics"][-1], flush=True)\n    del model, opt\n    if DEV == "cuda": torch.cuda.empty_cache()\n\n\nfor C in QUEUE:\n    try:\n        run_one(C)\n    except Exception as e:  # one failed config must not cost the rest of the queue\n        import traceback; traceback.print_exc()\n        with open(os.path.join(out_dir, C["runId"] + ".error.txt"), "w") as f: f.write(repr(e))\n        if DEV == "cuda": torch.cuda.empty_cache()\nprint("QUEUE DONE", flush=True)\n'
open('/tmp/worker.py', 'w').write(WORKER_SRC)


In [ ]:

t = time.time()
r = subprocess.run([sys.executable, "/tmp/prep.py", "/tmp/cache", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "prep failed"
print("prep wall", round(time.time() - t, 1))


In [ ]:
QUEUES = [[{'vocab': 4096, 'd': 512, 'layers': 9, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'micro': 32, 'saveFinal': True, 'runId': 'lltTinyStories-T2-size30m-s0', 'label': 'ablation:size=30M'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.01, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'stopIfDiverged': True, 'runId': 'lltTinyStories-T2-broken-lr1e-2-s0', 'label': 'broken:lr=1e-2'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.03, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'stopIfDiverged': True, 'runId': 'lltTinyStories-T2-broken-lr3e-2-s0', 'label': 'broken:lr=3e-2'}], [{'vocab': 4096, 'd': 192, 'layers': 5, 'heads': 6, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'saveFinal': True, 'runId': 'lltTinyStories-T2-size3m-s0', 'label': 'ablation:size=3M'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 1, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'saveAt': 2000, 'resumeTests': [0, 100], 'saveFinal': True, 'runId': 'lltTinyStories-T2-baseline-s1', 'label': 'baseline'}, {'vocab': 4096, 'd': 256, 'layers': 13, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'saveFinal': True, 'runId': 'lltTinyStories-T2-deep-s0', 'label': 'ablation:shape=deep'}, {'vocab': 4096, 'd': 448, 'layers': 4, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 200, 'steps': 3000, 'eval_every': 250, 'eval_batches': 40, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'prompts': ['Once upon a time', 'Tom and his dog went to the park. Tom', 'Lily wanted to bake a cake for her mom, but', 'The little bird could not fly. One day,', 'Max found a shiny key under his bed.'], 'saveFinal': True, 'runId': 'lltTinyStories-T2-wide-s0', 'label': 'ablation:shape=wide'}]]

procs = []
for gpu, q in enumerate(QUEUES):
    qp = f"/tmp/queue{gpu}.json"; json.dump(q, open(qp, "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(f"/kaggle/working/worker{gpu}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "/tmp/worker.py", qp, "/tmp/cache", "/kaggle/working"], env=env, stdout=log, stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(300)
    print(round((time.time() - T_SESSION) / 60, 1), "min", [p.poll() for p in procs], flush=True)
    for gpu in range(len(procs)):
        lines = open(f"/kaggle/working/worker{gpu}.log").read().splitlines()
        print(gpu, lines[-1][:200] if lines else "", flush=True)
print("exit codes", [p.returncode for p in procs])
json.dump({"sessionSec": round(time.time() - T_SESSION, 1)}, open("/kaggle/working/session.json", "w"))
print(sorted(os.listdir("/kaggle/working")))
